In [1]:
# We always start with a dataset to train on. Let's download the tiny shakespeare dataset
import urllib.request

urllib.request.urlretrieve('https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt', 'input.txt')

('input.txt', <http.client.HTTPMessage at 0x1110f5640>)

In [2]:
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [3]:
print("length of dataset in characters: ", len(text))

length of dataset in characters:  1115394


In [4]:
print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [5]:
chars = sorted((list(set(text))))
vocab_size = len(chars)
print(chars)
print(''.join(chars))
print(vocab_size)

['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']

 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [6]:
# This is our very simple tokenizer
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

print(encode('hii there'))
print(decode(encode('hii there')))

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


In [7]:
# Now encode entire dataset and store it in a jax array
import jax.numpy as jnp
data = jnp.array(encode(text), dtype=jnp.int64)
print(data.shape, data.dtype)
print(data[:1000])

/var/folders/dq/4lvhfhkn293_qhs1_jq0k86h0000gn/T/ipykernel_95462/3922664366.py:3: UserWarning: Explicitly requested dtype <class 'jax.numpy.int64'> requested in array is not available, and will be truncated to dtype int32. To enable more dtypes, set the jax_enable_x64 configuration option or the JAX_ENABLE_X64 shell environment variable. See https://github.com/google/jax#current-gotchas for more.
  data = jnp.array(encode(text), dtype=jnp.int64)


(1115394,) int32
[18 47 56 57 58  1 15 47 58 47 64 43 52 10  0 14 43 44 53 56 43  1 61 43
  1 54 56 53 41 43 43 42  1 39 52 63  1 44 59 56 58 46 43 56  6  1 46 43
 39 56  1 51 43  1 57 54 43 39 49  8  0  0 13 50 50 10  0 31 54 43 39 49
  6  1 57 54 43 39 49  8  0  0 18 47 56 57 58  1 15 47 58 47 64 43 52 10
  0 37 53 59  1 39 56 43  1 39 50 50  1 56 43 57 53 50 60 43 42  1 56 39
 58 46 43 56  1 58 53  1 42 47 43  1 58 46 39 52  1 58 53  1 44 39 51 47
 57 46 12  0  0 13 50 50 10  0 30 43 57 53 50 60 43 42  8  1 56 43 57 53
 50 60 43 42  8  0  0 18 47 56 57 58  1 15 47 58 47 64 43 52 10  0 18 47
 56 57 58  6  1 63 53 59  1 49 52 53 61  1 15 39 47 59 57  1 25 39 56 41
 47 59 57  1 47 57  1 41 46 47 43 44  1 43 52 43 51 63  1 58 53  1 58 46
 43  1 54 43 53 54 50 43  8  0  0 13 50 50 10  0 35 43  1 49 52 53 61  5
 58  6  1 61 43  1 49 52 53 61  5 58  8  0  0 18 47 56 57 58  1 15 47 58
 47 64 43 52 10  0 24 43 58  1 59 57  1 49 47 50 50  1 46 47 51  6  1 39
 52 42  1 61 43  5 50 50  1 46 39 

In [8]:
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

In [9]:
block_size = 8
train_data[:block_size+1]

Array([18, 47, 56, 57, 58,  1, 15, 47, 58], dtype=int32)

In [10]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context} the target: {target}")

when input is [18] the target: 47
when input is [18 47] the target: 56
when input is [18 47 56] the target: 57
when input is [18 47 56 57] the target: 58
when input is [18 47 56 57 58] the target: 1
when input is [18 47 56 57 58  1] the target: 15
when input is [18 47 56 57 58  1 15] the target: 47
when input is [18 47 56 57 58  1 15 47] the target: 58


In [11]:
import jax
from jax import random

key = jax.random.PRNGKey(42)
batch_size = 4
block_size = 8

def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = random.randint(key, (batch_size,), 0, len(data) - block_size)
    x = jnp.stack([data[i:i+block_size] for i in ix])
    y = jnp.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

print('----')

for b in range(batch_size): # batch dimension
    for t in range(block_size): # time dimension
        context = xb[b, :t+1]
        target = yb[b,t]
        print(f"when input is {context.tolist()} the target: {target}")

inputs:
(4, 8)
[[58  1 44 53 43 57  6  0]
 [ 1 57 43 43  1 58 46 43]
 [49  5 42  1 52 53 58  1]
 [39  1 50 43 61 42  1 42]]
targets:
(4, 8)
[[ 1 44 53 43 57  6  0 35]
 [57 43 43  1 58 46 43  1]
 [ 5 42  1 52 53 58  1 44]
 [ 1 50 43 61 42  1 42 39]]
----
when input is [58] the target: 1
when input is [58, 1] the target: 44
when input is [58, 1, 44] the target: 53
when input is [58, 1, 44, 53] the target: 43
when input is [58, 1, 44, 53, 43] the target: 57
when input is [58, 1, 44, 53, 43, 57] the target: 6
when input is [58, 1, 44, 53, 43, 57, 6] the target: 0
when input is [58, 1, 44, 53, 43, 57, 6, 0] the target: 35
when input is [1] the target: 57
when input is [1, 57] the target: 43
when input is [1, 57, 43] the target: 43
when input is [1, 57, 43, 43] the target: 1
when input is [1, 57, 43, 43, 1] the target: 58
when input is [1, 57, 43, 43, 1, 58] the target: 46
when input is [1, 57, 43, 43, 1, 58, 46] the target: 43
when input is [1, 57, 43, 43, 1, 58, 46, 43] the target: 1
when 

In [12]:
import equinox as eqx
import equinox.nn as nn
import jax.nn as F
import optax
from jax import vmap, jit, grad

class BigramLanguageModel(eqx.Module):
    token_embedding_table: jnp.array

    def __init__(self, vocab_size, key):
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size, key=key)

    def __call__(self, idx):
        # idx is a (B,T) tensor of integers
        logits = vmap(vmap(self.token_embedding_table))(idx)
        return logits
    
    def generate(self, key, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # Generate new random key
            key, _ = random.split(key)
            # get the predictions
            logits = self(idx)
            #focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = jax.nn.softmax(logits, axis=-1) # (B, C)
            # sample from the distribution
            def jax_multinomial(probabilities):
                return jax.random.choice(key, a=vocab_size, shape=(1, ), p=probabilities)
            idx_next = vmap(jax_multinomial)(probs)
            # idx_next = torch.multinomial(torch.tensor(probs.tolist()), num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = jnp.concat((idx, idx_next), axis=1) # (B, T+1)
        return idx

@eqx.filter_value_and_grad
def compute_loss(model, idx, targets):
    logits = model(idx)
    B, T, C = logits.shape
    logits = jnp.reshape(logits, (B*T, C))
    targets = jnp.reshape(targets, (B*T))
    loss = jnp.mean(optax.softmax_cross_entropy_with_integer_labels(logits, targets))
    return loss

    
m = BigramLanguageModel(vocab_size, key)
logits = m(xb)
print(logits.shape)
loss, grads = compute_loss(m, xb, yb)
print(loss)

print(decode(m.generate(key, idx = jnp.zeros((1, 1), dtype=jnp.int32), max_new_tokens=100)[0].tolist()))

(4, 8, 65)
4.564632

RQRySwP.WU&Ux :ADWPncj$dYE.DW,Ji!aikd:,GjcmoiaLlYPe:sEdHUfUxhnLX
w$,irmqOJJbDFBtT?fUkyFO3RRYVc
BiaGT


In [89]:
batch_size = 32
optim = optax.adamw(learning_rate=1e-3)
opt_state = optim.init(m)

@eqx.filter_jit
def make_step(model, x, y, opt_state):
    loss, grads = compute_loss(model, x, y)
    updates, opt_state = optim.update(grads, opt_state, m)
    model = eqx.apply_updates(model, updates)
    return loss, model, opt_state

for steps in range(10000):

    # sample batch of data
    xb, yb = get_batch('train')

    # evaluate the loss and update model params
    loss, m, opt_state = make_step(m, xb, yb, opt_state)
print(loss.item())

1.7465871572494507


In [91]:
print(decode(m.generate(key, idx = jnp.zeros((1, 1), dtype=jnp.int32), max_new_tokens=500)[0].tolist()))


Nukndelecl ls and;
Whighim
Bud, Duror'senoutecisong t of, ntht m
jo ry sof, lf t:
Tout l'sorel hindd sse: p?


ARWhim Gor'sp?
My, od, fory har r'set
Yo lf, o sp?
My htrd hthit: m ce. sig f, t sseneat f stow
Whar
WICK:
Bulig hthim
Whesteclod, od hisiretelout, haroud athaness?
WICK:
Tout: loukel m wang te troro ll hellelird;
Nury, r
My men hitelpat
Tory y lo olfory w
ARWhimete:
Nut s anofonclloou nor's?
Tore Gor bende hin w
Yonclof,
Nutellsparbeno fof sol ire llofowadd t Duddellecoo ls?
Whe hesule


## The mathematical trick in self-attention

---

In [13]:
B, T, C = 4, 8, 2
x = random.normal(key, (B, T, C))
x.shape

(4, 8, 2)

In [14]:
# One version
wei = jnp.tril(jnp.ones((T, T)))
wei = wei / jnp.sum(wei, axis=1, keepdims=True)
xbow = wei @ x  # (B, T, T) @ (B, T, C) ----> (B, T, C)

In [15]:
# Another version
tril = jnp.tril(jnp.ones((T, T)))
wei = jnp.where(tril == 0, -jnp.inf, 0)
wei = F.softmax(wei, axis=1)
xbow2 = wei @ x

In [16]:
jnp.allclose(xbow, xbow2)

Array(True, dtype=bool)

In [31]:
# Version 4: Self Attention
B, T, C = 4, 8, 32
x = random.normal(key, (B, T, C))

# lets see a single head perform self attention
head_size = 16
key_k, key_q = random.split(key)
key_ = nn.Linear(C, head_size, use_bias=False, key=key_k)
query = nn.Linear(C, head_size, use_bias=False, key=key_q)
value = nn.Linear(C, head_size, use_bias=False, key=key_q)

k = vmap(vmap(key_))(x)  # (B, T, head_size)
q = vmap(vmap(query))(x)  # (B, T, head_size)

wei = q @ jnp.transpose(k, axes=(0, 2, 1))  # (B, T, T)

tril = jnp.tril(jnp.ones((T, T)))
wei = jnp.where(tril == 0, -jnp.inf, wei)
wei = F.softmax(wei, axis=-1)

v = vmap(vmap(value))(x)  # (B, T, head_size)
out = wei @ v

out.shape

(4, 8, 16)

In [30]:
wei[0]

Array([[1.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        ],
       [0.7742495 , 0.22575043, 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        ],
       [0.30633488, 0.09025183, 0.60341334, 0.        , 0.        ,
        0.        , 0.        , 0.        ],
       [0.11261278, 0.24483077, 0.14224331, 0.5003131 , 0.        ,
        0.        , 0.        , 0.        ],
       [0.16678178, 0.19073178, 0.45656466, 0.0588486 , 0.12707317,
        0.        , 0.        , 0.        ],
       [0.07094556, 0.17847897, 0.1052039 , 0.6107223 , 0.01651821,
        0.01813102, 0.        , 0.        ],
       [0.05429098, 0.6218764 , 0.03484101, 0.0090417 , 0.11119799,
        0.03913375, 0.12961818, 0.        ],
       [0.08445949, 0.07266841, 0.13620576, 0.0184914 , 0.24857026,
        0.18933353, 0.19650526, 0.05376589]], dtype=float32)

```
step 0: train loss 4.3130, val loss 4.3492
step 500: train loss 2.3925, val loss 2.3039
step 1000: train loss 2.0612, val loss 2.0582
step 1500: train loss 1.9447, val loss 2.0334
step 2000: train loss 1.9784, val loss 1.8876
step 2500: train loss 1.8639, val loss 1.9899
step 3000: train loss 1.7717, val loss 1.8698
step 3500: train loss 1.8197, val loss 1.9480
step 4000: train loss 1.8265, val loss 1.8788
step 4500: train loss 1.7173, val loss 1.8130
Final Training Loss: 1.70048189163208
```
  
  
Apon thy cented you ressicious shall not slands' man!  
  
RICHARD:  
Let that did left derx met, noo eyer,  
Deaps forge true sign; thou whom my swared  
Is lack, the threm! 'Tybrokes and your ine it not chill of of Clabering?  
  
SICINIUS:  
I gain their Romees in a nefreed,  
Where I corses, against not in flows You thy ahts.  
  
GRENCE:  
Whrong is, could kinclorge diens.  
  
ESCALUS:  
She wither you have, thou not homraged  
fears the that Lader's you king give nor him a dishe sonst hat is your langs of a discapple in  